# 69 신규 표현의 고정 프로필 확인

왜: 67 검색 보완과 68 입력/지시문 개발선택이 새 표현에서도 유효한지 현재 SEQ65와 비교한다.

변경: U / DB3+기존 매칭 SEQ65 / DB4+명사 후보 투영+68 청킹별 선택 SEQ. 같은106문서·운영 청킹/색인·프로필·모델·스키마·출력한도는 고정한다. 질문 주제는 개발셋과 연관되며 신규 문서 독립 홀드아웃/독립 사람 gold라고 부르지 않는다.

**기본 ALLOW_NETWORK=0. 준비 셀만 오프라인 실행한다. 추출·임베딩·답변은 명시적 네트워크 허용과 main 독립리뷰 후에만 실행한다. 생성에는 실제 추가 fact 전수 원문 감사 gate가 별도로 필요하다.** 정답표·source registry는 자동조립기 입력이 아니다.

단계: 01 공통 준비 → 02 계약/manifest 동결 → 03 유료 추출 → 04 유료 임베딩 → 05 로컬 검색 → 06 자동 조립 → 07 입력 재생검증 → main 원문수락 → 08 gate 확인 → 09 유료 생성. 최종 화면 수동평가·질문 블록 bootstrap/선택/HTML은 main 후속 작업이다.

In [1]:
from pathlib import Path
import copy, importlib.util, json, os, sys, time, resource
from collections import Counter
from decimal import Decimal
import numpy as np
ROOT=Path.cwd() if (Path.cwd()/'notebooks').is_dir() else Path.cwd().parent
def module69(name, filename):
    spec=importlib.util.spec_from_file_location(name,ROOT/'notebooks/scripts'/filename)
    value=importlib.util.module_from_spec(spec);sys.modules[name]=value;spec.loader.exec_module(value);return value
V=module69('retrieval69_notebook','68_69_retrieval_runtime.py');T=V.T;R=T.R;A=V.A
F=module69('facts69_notebook','68_fact_input.py')
BASE=R.P.F.STAGES[69];NOTEBOOK=ROOT/'notebooks/69_repaired_sequential_confirmation.ipynb'
ALLOW_NETWORK=os.environ.get('ALLOW_NETWORK','0')=='1'
MODES=('U','SEQ65','SEQ_REPAIRED');PROFILES=tuple(R.PROFILES)
def freeze69(path,value): R.X.frozen(path,value)
def raw69(path):return T.E.file_digest(Path(path))
def provider_gate69():
    if not ALLOW_NETWORK:raise RuntimeError('ALLOW_NETWORK=0: 유료 셀은 main 독립리뷰 뒤 명시적으로 허용해야 합니다.')
    T.guard_manifest(T.read(BASE/'execution_manifest.json'))
    if T.budget_ledger()['unknown_cost_records']:raise RuntimeError('새 사용량 미확인: 추가 API 금지')
def resources69(start,cpu,**extra):
    return dict(segment_wall_seconds=time.perf_counter()-start,segment_cpu_seconds=time.process_time()-cpu,
        process_lifetime_peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
        ledger=T.budget_ledger(),first_token_seconds=None,browser_display_seconds=None,
        unmeasured=['TTFT','browser_display','actual_billed_invoice'],**extra)
def planned_generation_files69():
    return ['questions.json','question_contract.json','question_freeze.json','configuration_contract.json',
        'execution_manifest.json','parser_results.json','embedding_contract.json','query_vectors.npy',
        'source_inventory.json','retrieval_results.json','packages.json','jobs.json','answer_payload_contract.json',
        'additional_fact_inventory.json','planned_input_verification.json']
def checked_jobs69():
    manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
    jobs=T.read(BASE/'jobs.json')['records']
    expected={(q['query_id'],p,m,n) for q in T.read(BASE/'questions.json')['queries'] for p in PROFILES for m in MODES for n in (1,2,3)}
    assert len(jobs)==432 and {(j['query_id'],j['profile'],j['mode'],j['repeat'])for j in jobs}==expected
    assert len({j['identity']for j in jobs})==432
    assert all(j['identity']==f"{j['query_id']}:{j['profile']}:{j['mode']}:r{j['repeat']}"for j in jobs)
    assert set(j['identity']for j in jobs)==set(manifest['allowed_identities']['answer'])
    assert A.payload_contract(jobs,manifest)==T.read(BASE/'answer_payload_contract.json')
    return manifest,jobs

def load_vectors69(queries,base=None):
    base=BASE if base is None else base
    request=dict(model='text-embedding-3-small',input=[q['question']for q in queries],dimensions=1536)
    request_sha=R.P.sha(request);path=base/'api_cache'/('embedding_batch_'+request_sha+'.json')
    contract=T.read(base/'embedding_contract.json');stored=T.read(path)
    assert contract['request_sha256']==request_sha and contract['questions_sha256']==R.P.sha(queries)
    assert contract['raw_file_sha256']==R.P.sha(path.read_bytes())
    assert stored['request']==request and stored['status']=='ok'
    assert stored['usage']==stored['raw_response']['usage']
    ordered=sorted(stored['raw_response']['data'],key=lambda item:item['index'])
    assert [item['index']for item in ordered]==list(range(len(queries)))
    vectors=np.load(base/'query_vectors.npy',allow_pickle=False)
    assert vectors.shape==(len(queries),1536) and np.isfinite(vectors).all()
    assert np.array_equal(vectors,np.asarray([item['embedding']for item in ordered],dtype=np.float32))
    assert contract['matrix_sha256']==R.P.sha(vectors.tobytes())
    return vectors

def reconstruct_additional_inventory69(jobs,searches,facts):
    # inventory를 읽지 않고 실제 payload의 원retrieval prefix 뒤 JSON을 재추출한다.
    by_search={r['identity']:r for r in searches};assert len(by_search)==len(searches)==len(jobs)
    assert len({j['identity']for j in jobs})==len(jobs) and {j['identity']for j in jobs}==set(by_search)
    found={}
    for job in jobs:
        row=by_search[job['fixed_search_identity']]
        assert job['identity']==row['identity'] and job['retrieval_row_sha256']==R.P.sha(row)
        assert all(job[k]==row[k]for k in('query_id','question','profile','mode','repeat'))
        before=row['result'];after=job['result'];prefix=before['evidence']
        assert after['evidence'][:len(prefix)]==prefix
        assert {k:v for k,v in before.items()if k!='evidence'}=={k:v for k,v in after.items()if k!='evidence'}
        suffix=after['evidence'][len(prefix):]
        if job['mode']!='SEQ_REPAIRED':
            assert after==before
            continue
        cards={c['card_key']:c for c in before['cards']};assert len(cards)==len(before['cards'])
        seen=set()
        for evidence in suffix:
            canonical=json.loads(evidence['text']);assert isinstance(canonical,list) and len(canonical)==1
            item=canonical[0];fid=item['fact_id'];fact=facts[fid];card=item['card_key'];owner=cards[card]
            assert fid not in seen;seen.add(fid)
            assert fact['document_id']==card==evidence['card_key'] and evidence['chunk_id']=='s68:'+fid
            assert evidence['card_name']==item['card_name']==owner['card_name'] and evidence['issuer']==item['issuer']==owner['issuer']
            assert fid in row['routing']['matching_fact_ids'].get(card,[])
            links=[]
            for group,requirements in row['routing'].get('requirement_fact_ids',{}).get(card,{}).items():
                for index,matches in enumerate(requirements):
                    if fid in matches:links.append(dict(group=int(group),requirement=index,requirement_value=copy.deepcopy(row['routing']['plan']['groups'][int(group)][index])))
            expected=dict(card_key=card,card_name=owner['card_name'],issuer=owner['issuer'],fact_id=fid,ordinal=fact['ordinal'],
                **{k:copy.deepcopy(fact[k])for k in('fields','derived','relation_scope','field_evidence','source_relative_path','source_path_sha256','source_pdf_sha256')},requirement_links=links)
            assert item==expected and evidence['text']==F.render_json([item])
            entry=found.setdefault(fid,dict(fact_id=fid,card_key=card,database_fact_sha256=R.P.sha(fact),database_fact=copy.deepcopy(fact),occurrences=[]))
            entry['occurrences'].append(dict(identity=job['identity'],chunk_id=evidence['chunk_id'],text_sha256=R.P.sha(evidence['text']),canonical_record=item,canonical_record_sha256=R.P.sha(item)))
    return [found[fid]for fid in sorted(found)]

def validate_additional_inventory69(jobs,searches,facts,inventory):
    reconstructed=reconstruct_additional_inventory69(jobs,searches,facts)
    assert inventory['records']==reconstructed and inventory['unique_fact_count']==len(reconstructed)
    return R.P.sha(reconstructed)

def preparation_regressions69(queries,facts):
    import tempfile
    def write_fixture(path,value):path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n')
    checks=[]
    def rejected(label,call):
        try:call()
        except (AssertionError,KeyError,ValueError,RuntimeError):checks.append(label)
        else:raise AssertionError('변조를 거절하지 않음: '+label)
    with tempfile.TemporaryDirectory(prefix='69_embedding_fixture_',dir='/tmp')as directory:
        base=Path(directory);(base/'api_cache').mkdir();matrix=np.zeros((len(queries),1536),dtype=np.float32);np.save(base/'query_vectors.npy',matrix)
        request=dict(model='text-embedding-3-small',input=[q['question']for q in queries],dimensions=1536);request_sha=R.P.sha(request)
        write_fixture(base/'api_cache/embedding_batch_unrelated_failed.json',dict(status='failed',request={'input':['다른 질문']}))
        path=base/'api_cache'/('embedding_batch_'+request_sha+'.json');usage={'prompt_tokens':24,'total_tokens':24}
        write_fixture(path,dict(status='ok',request=request,usage=usage,raw_response=dict(usage=usage,data=[dict(index=i,embedding=matrix[i].tolist())for i in range(len(queries))])))
        contract=dict(request_sha256=request_sha,raw_file_sha256=R.P.sha(path.read_bytes()),questions_sha256=R.P.sha(queries),matrix_sha256=R.P.sha(matrix.tobytes()))
        write_fixture(base/'embedding_contract.json',contract);assert np.array_equal(load_vectors69(queries,base),matrix);checks.append('exact_request_raw_selected_with_unrelated_failure_present')
        for key in('request_sha256','questions_sha256','raw_file_sha256','matrix_sha256'):
            changed=copy.deepcopy(contract);changed[key]='bad';write_fixture(base/'embedding_contract.json',changed)
            rejected('embedding_contract_'+key,lambda:load_vectors69(queries,base))
        write_fixture(base/'embedding_contract.json',contract)
    fact=next(iter(facts.values()));fid=fact['fact_id'];card=fact['document_id'];owner=dict(card_key=card,card_name='fixture',issuer='fixture')
    original=dict(**owner,chunk_id='u_fixture',text='원래 근거')
    row=dict(identity='fixture:profile:SEQ_REPAIRED:r1',query_id='fixture',question='fixture',profile='profile',mode='SEQ_REPAIRED',repeat=1,
        result=dict(cards=[owner],evidence=[original]),routing=dict(matching_fact_ids={card:[fid]},requirement_fact_ids={card:{'0':[[fid]]}},plan={'groups':[[{'targets':['fixture']}]]}))
    canonical=F.canonical_records(row['result'],row['routing'],facts)[0]
    suffix=dict(**owner,chunk_id='s68:'+fid,text=F.render_json([canonical]))
    job={k:copy.deepcopy(row[k])for k in('identity','query_id','question','profile','mode','repeat')}
    job.update(fixed_search_identity=row['identity'],retrieval_row_sha256=R.P.sha(row),result=dict(cards=[owner],evidence=[original,suffix]))
    records=reconstruct_additional_inventory69([job],[row],facts);inventory=dict(records=records,unique_fact_count=len(records))
    assert validate_additional_inventory69([job],[row],facts,inventory)==R.P.sha(records);checks.append('complete_actual_payload_inventory')
    for kind in('missing_fact','owner','text_sha','canonical','occurrence'):
        changed=copy.deepcopy(inventory)
        if kind=='missing_fact':changed.update(records=[],unique_fact_count=0)
        elif kind=='owner':changed['records'][0]['card_key']='wrong'
        elif kind=='text_sha':changed['records'][0]['occurrences'][0]['text_sha256']='bad'
        elif kind=='canonical':changed['records'][0]['occurrences'][0]['canonical_record']['ordinal']=-1
        else:changed['records'][0]['occurrences']=[]
        rejected('actual_payload_inventory_'+kind,lambda:validate_additional_inventory69([job],[row],facts,changed))
    changed=copy.deepcopy(job);changed['result']['evidence'][0]['text']='바뀐 원래 근거'
    rejected('original_U_prefix',lambda:validate_additional_inventory69([changed],[row],facts,inventory))
    return checks

def require_source_gate69():
    manifest,jobs=checked_jobs69()
    path=BASE/'pre_answer_acceptance.json'
    if not path.exists():raise RuntimeError('main의 추가fact 전수 원문 감사/수락 gate가 없습니다. 답변 생성 금지.')
    gate=T.read(path);T.E.verify_file_map(gate['files_sha256'],ROOT)
    assert gate['status']=='accepted_source_and_input_assembly_before_generation'
    expected={str((BASE/name).relative_to(ROOT)):raw69(BASE/name)for name in planned_generation_files69()}
    assert all(gate['files_sha256'].get(name)==digest for name,digest in expected.items())
    assert gate['jobs_raw_sha256']==raw69(BASE/'jobs.json')
    assert gate['execution_manifest_sha256']==R.P.sha(manifest)
    inventory=T.read(BASE/'additional_fact_inventory.json')
    assert gate['additional_fact_inventory_raw_sha256']==raw69(BASE/'additional_fact_inventory.json')
    reconstructed_sha=validate_additional_inventory69(jobs,T.read(BASE/'retrieval_results.json')['records'],R.R.fact_map(ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4'),inventory)
    assert gate['reconstructed_additional_inventory_sha256']==reconstructed_sha
    assert T.read(BASE/'planned_input_verification.json')['reconstructed_additional_inventory_sha256']==reconstructed_sha
    assert gate['reviewed_fact_sha256_by_id']=={r['fact_id']:r['database_fact_sha256']for r in inventory['records']}
    assert gate['all_transmitted_unique_facts_reviewed'] is True and gate['confirmed_source_conflicts']==0
    assert gate['unreviewed_fact_ids']==[] and gate['manual_gold_in_assembler'] is False
    review_path=ROOT/gate['source_review_path']
    assert gate['files_sha256'][gate['source_review_path']]==raw69(review_path)==gate['source_review_raw_sha256']
    source_review=T.read(review_path)
    assert source_review['reconstructed_additional_inventory_sha256']==reconstructed_sha
    assert source_review['reviewed_fact_sha256_by_id']==gate['reviewed_fact_sha256_by_id']
    return manifest,jobs,gate
print('공통','런타임 준비, ALLOW_NETWORK=',ALLOW_NETWORK,'; API 호출 없음')


공통 런타임 준비, ALLOW_NETWORK= False ; API 호출 없음


## 02 오프라인 준비와 계약 동결

새 질문과 직접요구만 고정한다. 정답 카드/조건표를 parser나 assembler에 주지 않는다. 아래 manifest의 실행 phase hash는 이 노트북의 모든 `exp69-exec-` 코드셀 작성이 끝난 뒤 계산된다. 파일이 달라지면 재개를 거절한다. 과거 raw/API/DB는 읽기 전용이다.

In [2]:
start=time.perf_counter();cpu=time.process_time()
queries=T.read(BASE/'questions.json')['queries'];rubric=T.read(BASE/'question_contract.json')
config=T.read(BASE/'configuration_contract.json')
selected68=T.read(ROOT/config['selection68_path']);selected67=T.read(ROOT/config['selection67_path'])
assert selected68['status']=='accepted_development_input_prompt_candidate_for69'
assert selected67['parser_variant']=='old' and selected67['matcher']=='projected'
assert raw69(ROOT/config['selection68_path'])==config['selection68_raw_sha256']
assert raw69(ROOT/config['selection67_path'])==config['selection67_raw_sha256']
T.E.verify_file_map(selected68['files_sha256'],ROOT);T.E.verify_file_map(selected67['files_sha256'],ROOT)
selected_profiles={r['profile']:{k:r[k]for k in('input_arm','prompt_variant','mode')}for r in selected68['records']}
assert selected_profiles==config['profiles']
assert Counter(q['subgroup']for q in queries)=={'short_recommendation':8,'compound':6,'named_detail':4,'general':4,'insufficient_information':2}
assert len(queries)==24 and len({q['query_id']for q in queries})==len({q['question']for q in queries})==24
assert [r['query_id']for r in rubric['records']]==[q['query_id']for q in queries]
assert all(r['question']==q['question']for r,q in zip(rubric['records'],queries,strict=True))
# 과거 원질문의 정확 문자열 중복만 검사한다. 의미군 재사용은 공개한다.
previous_paths=sorted(p for p in (ROOT/'notebooks/data').glob('*/questions.json') if p!=BASE/'questions.json' and p.parent.name[:2].isdigit() and 30<=int(p.parent.name[:2])<=68)
previous_paths.append(ROOT/'notebooks/data/66_visible_answer_policy/question_contract.json')
def question_texts69(value):
    if isinstance(value,dict):
        if isinstance(value.get('question'),str):yield value['question']
        for key,item in value.items():
            if key!='question':yield from question_texts69(item)
    elif isinstance(value,list):
        for item in value:yield from question_texts69(item)
old={text for path in previous_paths for text in question_texts69(T.read(path))}
assert not (old & {q['question']for q in queries}), '기존 원질문 정확 중복'
# 두 구조화 버전은 같은106카드 카탈로그/원문 소유자를 사용한다.
db3=R.M.DB_NEW;db4=ROOT/'notebooks/data/67_structured_retrieval_repair/db_v4'
docs3=[json.loads(line)for line in(db3/'documents.jsonl').read_text().splitlines()]
docs4=[json.loads(line)for line in(db4/'documents.jsonl').read_text().splitlines()]
assert docs3==docs4 and len(docs3)==106
preparation_regression_checks=preparation_regressions69(queries,R.R.fact_map(db4))
assert len(preparation_regression_checks)==12
assert T.OLD_PARSER_PROMPT==R.R.NEW_PROMPT
source_paths=set(R.M.source_hashes())|{'notebooks/scripts/'+s for s in ['67_69_execution_runtime.py','67_query_repair.py','67_db_repair.py','65_scope_repair.py','68_fact_input.py','68_69_answer_runtime.py','68_69_retrieval_runtime.py']}
sources={name:raw69(ROOT/name)for name in sorted(source_paths)}
critical={}
for directory in(db3,db4):
    for name in('structured.sqlite','facts.jsonl','documents.jsonl'):
        path=directory/name;critical[str(path.relative_to(ROOT))]=raw69(path)
for name in('questions.json','question_contract.json','configuration_contract.json'):
    critical[str((BASE/name).relative_to(ROOT))]=raw69(BASE/name)
for path in [ROOT/config['selection68_path'],ROOT/config['selection67_path'],ROOT/'notebooks/data/66_visible_answer_policy/policy.json',ROOT/'notebooks/data/68_structured_evidence_generation/source_union_manifest.json']:
    critical[str(path.relative_to(ROOT))]=raw69(path)
critical.update(selected68['files_sha256']);critical.update(selected67['files_sha256'])
freeze_path=BASE/'question_freeze.json'
freeze69(freeze_path,dict(status='frozen_before69_extraction_retrieval_answers',files_sha256={str((BASE/n).relative_to(ROOT)):raw69(BASE/n)for n in('questions.json','question_contract.json','configuration_contract.json')},
    selection68_raw_sha256=config['selection68_raw_sha256'],created_at_epoch=T.read(BASE/'questions.json')['created_at_epoch'],
    new_question_texts=24,new_independent_documents=False,independent_human_gold=False,paid_calls_at_creation=0,
    exact_duplicate_previous_questions=0,duplicate_check_files_sha256={str(p.relative_to(ROOT)):raw69(p)for p in previous_paths},manual_gold_not_model_input=True))
critical[str(freeze_path.relative_to(ROOT))]=raw69(freeze_path)
phase={str(NOTEBOOK.relative_to(ROOT)):dict(prefix='exp69-exec-',sha256=T.phase_code_sha(NOTEBOOK,'exp69-exec-'))}
parser_logical=[f"{q['query_id']}:parser:r{n}"for q in queries for n in(1,2,3)]
physical=[T.attempt_identity(i,a)for i in parser_logical for a in(1,2)];by_qid={q['query_id']:q for q in queries}
parser_manifest=dict(stage=690,parser_variant='old',model='gpt-6-luna',reasoning='medium',max_output_tokens=3000,
    source_sha256=sources,critical_data_sha256=critical,notebook_phase_code_sha256=phase,
    allowed_identities=dict(parser=physical,answer=[]),logical_identity_by_physical={T.attempt_identity(i,a):i for i in parser_logical for a in(1,2)},
    allowed_request_by_identity={i:R.P.sha([dict(role='user',content=by_qid[i.split(':')[0]]['question'])])for i in physical},
    parser_instructions_sha256=R.P.sha(T.OLD_PARSER_PROMPT),parser_schema_sha256=R.P.sha(R.Q.QueryPlan.model_json_schema()),
    logical_parser_requests=72,maximum_parser_requests=144,maximum_answer_requests=0,maximum_embedding_requests=0,maximum_physical_requests=144,stage_cost_cap_usd='8')
freeze69(R.P.F.STAGES[690]/'execution_manifest.json',parser_manifest)
identities=[f"{q['query_id']}:{p}:{m}:r{n}"for q in queries for p in PROFILES for m in MODES for n in(1,2,3)]
prompt_variant={i:config['profiles'][i.split(':')[1]]['prompt_variant']if i.split(':')[2]=='SEQ_REPAIRED'else'old'for i in identities}
prompts=R.G.C.operating_prompts();old_manifest=T.read(R.B65/'execution_manifest.json')
manifest=dict(stage=69,model='gpt-6-luna',reasoning='medium',source_sha256=sources,critical_data_sha256=critical,
    notebook_phase_code_sha256=phase,notebook_code_sha256={},data_sha256=old_manifest['data_sha256'],
    allowed_identities=dict(parser=[],answer=identities),maximum_parser_requests=0,maximum_answer_requests=864,maximum_embedding_requests=2,
    maximum_physical_requests=866,stage_cost_cap_usd='8',global_maximum_physical_requests_69=1010,logical_answers=432,
    prompt_variant_by_identity=prompt_variant,answer_instructions_sha256_by_identity={i:[R.P.sha(T.transform_prompt(p,prompt_variant[i]))for p in prompts]for i in identities},
    parameters=old_manifest['parameters'],scope_version='catalog_short_identifier_followup65',original_U_union=False,profile_retuning=False,
    configuration_contract_raw_sha256=raw69(BASE/'configuration_contract.json'),per_mode_versions=config)
freeze69(BASE/'execution_manifest.json',manifest)
freeze69(BASE/'source_inventory.json',V.inventory())
freeze69(BASE/'source_reference_link.json',dict(evaluation_only=True,not_model_input=True,global_library_path='notebooks/data/68_structured_evidence_generation/source_registry.json',
    source_union_manifest_path='notebooks/data/68_structured_evidence_generation/source_union_manifest.json',
    source_union_raw_sha256=raw69(ROOT/'notebooks/data/68_structured_evidence_generation/source_union_manifest.json'),manual_gold_not_assembler_input=True))
freeze69(BASE/'offline_preparation.json',dict(status='prepared_without_paid_cells_or_search',questions=24,parser_logical=72,answers_logical=432,physical_caps=dict(parser=144,answer=864,embedding=2,total=1010),
    exact_duplicate_questions=0,catalog_documents=106,questions_created_after_observed68_selection=True,
    preparation_regression_checks=preparation_regression_checks,
    question_freeze_raw_sha256=raw69(freeze_path),execution_manifest_raw_sha256=raw69(BASE/'execution_manifest.json'),
    notebook_phase_code_sha256=phase,network_default=False,provider_calls_during_preparation=0,
    source_gate_not_created=True,answer_files_not_created=True))
print('오프라인 준비완료: 질문24/공유추출72/신규답변432, API0. 다음은03추출 셀이며 main리뷰 후에만 실행.')
print(resources69(start,cpu,phase='offline_preparation'))


오프라인 준비완료: 질문24/공유추출72/신규답변432, API0. 다음은03추출 셀이며 main리뷰 후에만 실행.
{'segment_wall_seconds': 1.4749351739883423, 'segment_cpu_seconds': 1.4741420389999997, 'process_lifetime_peak_rss_kib': 212188, 'ledger': {'record_count': 825, 'known_priced_records': 825, 'unknown_cost_records': 0, 'duplicate_saved_responses': 0, 'known_cost_usd': '2.168031040', 'tokens': {'input_tokens': 4516535, 'output_tokens': 653147, 'ordinary_input_tokens': 2472, 'cached_input_tokens': 3320687, 'cache_write_tokens': 1193376, 'embedding_tokens': 878}, 'billed_cost_usd': None, 'budget_debit_usd': '2.168031040', 'new_known_API_cost_usd': '0.509217130', 'historical_reserved_usd': '0.006234500', 'note': 'known_cost_usd is the legacy guard compatibility field and includes approved historical reservations; not billed invoice'}, 'first_token_seconds': None, 'browser_display_seconds': None, 'unmeasured': ['TTFT', 'browser_display', 'actual_billed_invoice'], 'phase': 'offline_preparation'}


## 03 유료 추출 — 이번 준비 작업에서 실행하지 않음

새 질문24×3의 old parser 72논리를 한 번씩 생성한다. 두 SEQ와 두 프로필은 같은 실제 parser row를 참조한다. 기술실패도 보존하며 의미가 마음에 들지 않는다는 이유로 재추출하지 않는다. 원문 기준/수동 rubric은 API에 보내지 않는다.

In [ ]:
provider_gate69();start=time.perf_counter();cpu=time.process_time();parsers={}
for q in queries:
    for repeat in(1,2,3):
        row=T.extract_query(690,q,repeat,allow_network=ALLOW_NETWORK);parsers[q['query_id'],repeat]=row
    print('추출',q['query_id'],len(parsers),'/72',flush=True)
assert len(parsers)==72
freeze69(BASE/'parser_results.json',dict(records=list(parsers.values()),parser_variant='old',shared_between_both_SEQ_and_profiles=True))
T.write(BASE/'parser_resources.json',resources69(start,cpu,logical_parser_rows=72,
    physical_events=sum(len(r['events'])for r in parsers.values()),
    inclusive_parser_ready_seconds=sum(r['parser_ready_wall_seconds']for r in parsers.values())))


추출 C01 3 /72
추출 C02 6 /72
추출 C03 9 /72
추출 C04 12 /72
추출 C05 15 /72
추출 C06 18 /72
추출 C07 21 /72
추출 C08 24 /72
추출 C09 27 /72
추출 C10 30 /72
추출 C11 33 /72
추출 C12 36 /72
추출 C13 39 /72
추출 C14 42 /72
추출 C15 45 /72
추출 C16 48 /72
추출 C17 51 /72
추출 C18 54 /72
추출 C19 57 /72
추출 C20 60 /72
추출 C21 63 /72
추출 C22 66 /72
추출 C23 69 /72
추출 C24 72 /72


## 04 유료 임베딩 — 이번 준비 작업에서 실행하지 않음

24원질문을 한 배치로 만든다. 기존 배치 함수의 예산 경로만 호출 동안 canonical 8달러 원장에 연결하고 `finally`로 복구한다. 새 사용량 미확인/중단 raw는 자동 우회하지 않는다. provider는 직렬이다.

In [ ]:
provider_gate69();start=time.perf_counter();cpu=time.process_time()
manifest=T.read(BASE/'execution_manifest.json');previous_ledger=R.X.budget_upper_ledger
try:
    R.X.budget_upper_ledger=T.budget_ledger
    vectors=R.embedding_batch(69,queries,manifest,allow_network=ALLOW_NETWORK)
finally:
    R.X.budget_upper_ledger=previous_ledger
assert vectors.shape==(24,1536)
if T.budget_ledger()['unknown_cost_records']:raise RuntimeError('임베딩 사용량 미확인: 추가 요청 금지')
T.write(BASE/'embedding_resources.json',resources69(start,cpu,batches=len(list((BASE/'api_cache').glob('embedding_batch_*.json'))),vectors=24))


## 05 로컬 검색

구조화 양성이면 해당 카드범위 U1회, 0건/추출실패면 원 scope U1회. 범위 밖 global U를 합치지 않는다. 같은 최종 scope의 검색 재사용은 허용하되 routing·identity·seal은 각각 남긴다. 현재SEQ65와 수정SEQ를 명칭에 명시한다.

In [ ]:
start=time.perf_counter();cpu=time.process_time();manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
parsers={(r['query_id'],r['repeat']):r for r in T.read(BASE/'parser_results.json')['records']}
parser_manifest=T.read(R.P.F.STAGES[690]/'execution_manifest.json')
for q in queries:
    for n in(1,2,3):T.validate_parser_checkpoint(parsers[q['query_id'],n],q,parser_manifest,R.P.F.STAGES[690])
vectors=load_vectors69(queries)
facts3=R.R.fact_map(db3);facts4=R.R.fact_map(db4);documents=docs3
runtime=V.Runtime(69);searches=[];routing_pairs=[]
for index,(q,vector)in enumerate(zip(queries,vectors,strict=True)):
    original=R.S.original_scope(q['question'],documents)
    for repeat in(1,2,3):
        parser=parsers[q['query_id'],repeat]
        route65=T.K.route(R,parser['plan'],original,db3,facts3,matcher='baseline',parser_failed=parser['status']!='ok')
        route67=T.K.route(R,parser['plan'],original,db4,facts4,matcher='projected',parser_failed=parser['status']!='ok')
        routes=dict(U=dict(route='U',scope=original['card_keys'],reason='independent_unstructured',db_attempted=False),SEQ65=route65,SEQ_REPAIRED=route67)
        routing_pairs.append(dict(query_id=q['query_id'],repeat=repeat,parser_row_sha256=R.P.sha(parser),original_scope=original,SEQ65=route65,SEQ_REPAIRED=route67))
        for profile in(PROFILES if(index+repeat)%2 else tuple(reversed(PROFILES))):
            shift=(index+repeat)%3
            for mode in MODES[shift:]+MODES[:shift]:
                row=V.retrieve(runtime,q,vector,profile,repeat,routes[mode],original,mode=mode,parser_binding=R.P.sha(parser));searches.append(row)
    print('검색',index+1,'/24',flush=True)
assert len(searches)==432 and len({r['identity']for r in searches})==432
freeze69(BASE/'retrieval_results.json',dict(records=searches,original_U_union=False))
freeze69(BASE/'routing_comparisons.json',dict(records=routing_pairs,shared_parser_rows=72))
values=dict(runtime.resources());values.pop('peak_rss_kib',None)
T.write(BASE/'retrieval_resources.json',resources69(start,cpu,logical_searches=432,search_resources=values,profiles_retuned=False))


검색 1 /24
검색 2 /24
검색 3 /24
검색 4 /24
검색 5 /24
검색 6 /24
검색 7 /24
검색 8 /24
검색 9 /24
검색 10 /24
검색 11 /24
검색 12 /24
검색 13 /24
검색 14 /24
검색 15 /24
검색 16 /24
검색 17 /24
검색 18 /24
검색 19 /24
검색 20 /24
검색 21 /24
검색 22 /24
검색 23 /24
검색 24 /24


## 06 자동 조립과432개 fresh 답변 작업 동결

U/현재SEQ65는 추가 사실이 없다. 수정SEQ만 자동 same-card matching fact의 B(JSON)를 원U 뒤에 붙인다. 수동 정답/원문 registry는 조립기에 전달하지 않는다. 두 표현의 페이로드 제한이 넘으면 기존68 쌍A복귀 규칙을 그대로 보존한다. 실제 추가 전송된 고유fact만 별도 감사목록에 남긴다. 알려진 Monthly150/50 충돌을 선택적으로 바꾸거나 숨기지 않는다.

In [ ]:
start=time.perf_counter();cpu=time.process_time();manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
searches=T.read(BASE/'retrieval_results.json')['records'];facts4=R.R.fact_map(db4);jobs=[];packages=[];additional={}
for row in searches:
    q=next(q for q in queries if q['query_id']==row['query_id']);mode=row['mode'];result=copy.deepcopy(row['result']);package=None
    if mode=='SEQ_REPAIRED':
        package=F.assemble(q['question'],row['result'],row['routing'],facts4)
        result=package['results'][config['profiles'][row['profile']]['input_arm']]
        packages.append(dict(identity=row['identity'],query_id=row['query_id'],profile=row['profile'],repeat=row['repeat'],original_U_result_sha256=R.P.sha(row['result']),**{k:v for k,v in package.items()if k!='results'}))
        assert result['cards']==row['result']['cards'] and result['evidence'][:len(row['result']['evidence'])]==row['result']['evidence']
        for evidence in result['evidence'][len(row['result']['evidence']):]:
            canonical=json.loads(evidence['text']);assert len(canonical)==1;item=canonical[0];fid=item['fact_id'];fact=facts4[fid]
            assert fact['document_id']==evidence['card_key']==item['card_key']
            entry=additional.setdefault(fid,dict(fact_id=fid,card_key=fact['document_id'],database_fact_sha256=R.P.sha(fact),database_fact=fact,occurrences=[]))
            entry['occurrences'].append(dict(identity=row['identity'],chunk_id=evidence['chunk_id'],text_sha256=R.P.sha(evidence['text']),canonical_record=item,canonical_record_sha256=R.P.sha(item)))
    jobs.append(dict(identity=row['identity'],query_id=row['query_id'],question=row['question'],profile=row['profile'],mode=mode,repeat=row['repeat'],result=result,retrieval_status=row['retrieval_status'],
        fixed_search_identity=row['identity'],retrieval_row_sha256=R.P.sha(row),package_applicable=package['applicable']if package else False))
assert len(jobs)==432 and len(packages)==144
freeze69(BASE/'packages.json',dict(records=packages,automatic_only=True,no_U_eviction=True))
freeze69(BASE/'jobs.json',dict(records=jobs))
freeze69(BASE/'additional_fact_inventory.json',dict(status='transmitted_unique_facts_pending_main_full_source_review',records=[additional[k]for k in sorted(additional)],
    unique_fact_count=len(additional),manual_gold_in_assembler=False,source_review_complete=False,
    inherited_known_conflict='DiscountPlan+ Monthly원문150만원vsOCR/DB50만원. 실제추가목록포함여부와별개로원U충돌을보존한다. 특정fact수동교체/제외금지.'))
freeze69(BASE/'answer_payload_contract.json',A.payload_contract(jobs,manifest))
T.write(BASE/'assembly_resources.json',resources69(start,cpu,logical_jobs=432,automatic_packages=144,transmitted_unique_facts=len(additional)))
print('432jobs 동결. 답변 생성 전 실제 추가fact 원문 전수감사/main수락 필요.')


432jobs 동결. 답변 생성 전 실제 추가fact 원문 전수감사/main수락 필요.


## 07 입력 재생 검증 — source 수락을 만들지 않음

저장 typed plan을 동일 DB/matcher에 재실행하고 실제 검색→자동조립→serialized payload를 비교한다. SQL 시간만 비교에서 제외한다. source registry의 진실 여부는 이 검증으로 자동 통과시키지 않는다.

In [ ]:
start=time.perf_counter();cpu=time.process_time();manifest,jobs=checked_jobs69()
searches={r['identity']:r for r in T.read(BASE/'retrieval_results.json')['records']}
facts3=R.R.fact_map(db3);facts4=R.R.fact_map(db4)
for job in jobs:
    row=searches[job['fixed_search_identity']]
    assert job['retrieval_row_sha256']==R.P.sha(row)
    disk=BASE/'retrievals'/(row['identity']+'.json')
    assert T.read(disk)==row and T.read(disk.with_suffix('.seal.json'))['row_sha256']==R.P.sha(row)
    R.G.T.S.validate_scope(job['result'],row['original_scope']['card_keys']);R.G.T.S.validate_scope(job['result'],row['routing']['scope'])
    q=next(q for q in queries if q['query_id']==job['query_id']);parser=parsers[q['query_id'],job['repeat']]
    if job['mode']=='U':expected=row['result']
    else:
        repaired=job['mode']=='SEQ_REPAIRED';directory,fmap=(db4,facts4)if repaired else(db3,facts3)
        route=T.K.route(R,parser['plan'],row['original_scope'],directory,fmap,matcher='projected'if repaired else'baseline',parser_failed=parser['status']!='ok')
        assert {k:v for k,v in route.items()if k!='db_wall_seconds'}=={k:v for k,v in row['routing'].items()if k!='db_wall_seconds'}
        expected=F.assemble(q['question'],row['result'],route,facts4)['results'][config['profiles'][job['profile']]['input_arm']]if repaired else row['result']
    assert job['result']==expected
inventory=T.read(BASE/'additional_fact_inventory.json')
reconstructed_sha=validate_additional_inventory69(jobs,list(searches.values()),facts4,inventory)
freeze69(BASE/'planned_input_verification.json',dict(status='all432_planned_payloads_and_routes_verified_not_source_truth_acceptance',
    logical_jobs=432,shared_parser_rows=72,reconstructed_additional_inventory_sha256=reconstructed_sha,source_truth_gate_pending=True,manual_gold_in_assembler=False,
    jobs_raw_sha256=raw69(BASE/'jobs.json'),retrieval_results_raw_sha256=raw69(BASE/'retrieval_results.json'),
    payload_contract_raw_sha256=raw69(BASE/'answer_payload_contract.json'),additional_fact_inventory_raw_sha256=raw69(BASE/'additional_fact_inventory.json'),
    typed_SQL_replayed_both_DB_versions=True,original_scope_preserved=True,original_U_prefix_preserved=True,global_U_union=False))
T.write(BASE/'input_verification_resources.json',resources69(start,cpu,checked_jobs=432))
print('입력/소유/경로 재생검증 완료. source 충돌0 전수감사 gate는 main이 별도로 작성한다.')


## main 원문 감사 게이트 계약

main은 `additional_fact_inventory.json`의 **실제 전송된 모든 고유 fact**를 원PDF와 값·단위·대상·같은혜택 관계별로 대조한다. 개별 정답fact만 골라 바꾸거나 제외하지 않는다. 충돌이 남으면 새432답변을 진행하지 않고 원실험을 보존한다.

`pre_answer_acceptance.json`은 이 노트북이 만들지 않는다. 필수: status=`accepted_source_and_input_assembly_before_generation`, planned_generation_files69 전체 rawSHA를 포함한 files_sha256, jobs_raw_sha256, execution_manifest_sha256, additional_fact_inventory_raw_sha256, inventory와 정확히 같은 reviewed_fact_sha256_by_id, all_transmitted_unique_facts_reviewed=true, confirmed_source_conflicts=0, unreviewed_fact_ids=[], manual_gold_in_assembler=false, source_review_path/rawSHA. 또한 실제 jobs의 원retrieval prefix와 구조화 suffix에서 독립 재추출한 reconstructed_additional_inventory_sha256가 planned_input_verification·main gate·source review JSON 세 곳에서 모두 같아야 하며, source review의 reviewed_fact_sha256_by_id도 정확히 같아야 한다. source 검토의 미확정과 한계도 원문 감사자료에 명시한다.

## 08 답변 전 gate 확인 — 유료 호출 없음

In [ ]:
manifest,jobs,gate=require_source_gate69()
print('main source gate/current432jobs/payload/manifest 검증완료. 다음09셀만 실제답변생성.')


## 09 유료 답변 생성 — 이번 준비 작업에서 실행하지 않음

세 방식×2프로필×24질문×3회=432논리를 직렬 생성한다. 동일 입력이더라도 logical identity마다 fresh이며 반복 응답 복사 금지. 같은 identity의 실제 저장행 재개만 허용한다. 기술실패를 정답/제외로 바꾸지 않고 실제 raw·회계·분모를 보존한다.

In [ ]:
provider_gate69();manifest,jobs,gate=require_source_gate69();start=time.perf_counter();cpu=time.process_time()
expected_jobs={j['identity']:R.P.sha(j)for j in jobs};disk_jobs_sha=raw69(BASE/'jobs.json');source_gate_sha=raw69(BASE/'pre_answer_acceptance.json');rows=[]
for index,job in enumerate(jobs,1):
    if (expected_jobs.get(job.get('identity'))!=R.P.sha(job) or raw69(BASE/'jobs.json')!=disk_jobs_sha
            or raw69(BASE/'pre_answer_acceptance.json')!=source_gate_sha):
        raise RuntimeError('동결 jobs의 본문/metadata 변경: provider 전 중단')
    row=A.generate(69,job,allow_network=ALLOW_NETWORK);rows.append(row)
    if index%12==0:print('답변',index,'/432, 누적보수차감',T.budget_ledger()['budget_debit_usd'],flush=True)
freeze69(BASE/'execution_results.json',dict(records=rows,logical_answers=432,fresh_logical_identity_responses=True))
events=[e for r in rows for e in r['events']];roundtrips=[e.get('observed_call_wall_seconds')for e in events]
T.write(BASE/'answer_resources.json',resources69(start,cpu,logical_answers=432,physical_events=len(events),
    provider_roundtrip_wall_seconds=sum(roundtrips)if all(v is not None for v in roundtrips)else None,
    provider_roundtrip_measured_events=sum(v is not None for v in roundtrips),
    inclusive_answer_ready_wall_seconds=sum(r['answer_ready_wall_seconds']for r in rows),
    raw_journals='api_cache/*.json',new_response_reuse_across_repeats=False))
print('432행 생성완료. 이후 화면 수동평가/비교/보고는 main의 별도 단계이며 여기서자동정답채점하지 않는다.')


## 완료 조건과 보고 한계

실행 후 실제 raw 입력 복원·모든 표면 의미판정·직접요구/실제정확성/입력지원/추가설명 분리·실제추가주장 감사가 필요하다. 질문별3회 평균과 승패·그룹을 공개하고, 질문 블록 bootstrap10000/seed69/95%CI로 비교한다. 반복을 독립 질문으로 세지 않는다. 동률/불확실은 현재 유지. 결과를 본 뒤 수정하면 개발셋으로 재분류한다. 종합 설정 비교이므로 SQL만의 효과라고 부르지 않는다. 최종 집계·선택·HTML은 이 준비에서 만들지 않는다.

### 저장 JSON 순서를 사용하는 검증 보완
실제69 입력에서07의 typed SQL 값은 같았지만, 메모리 Pydantic dict와 저장 JSON의 키 순서 때문에 추가 근거 문자열 재조립 비교가 실패했다. 기존 유료 실행 phase와 원입력을 보존한다. 단계 재현 시07 대신 아래 검증 셀을 실행한다. 노트북 전체 Run All은 원래07 실패를 재현하므로 이번 완료 검증은 단계 실행이다. 아래 셀도 원문 충돌 수락을 대신하지 않는다.


In [ ]:
# Verification-only repair. Original paid phase, rows and payload bytes stay frozen.
def saved_route_for_replay69(fresh, saved):
    omit=lambda value:{k:v for k,v in value.items()if k!='db_wall_seconds'}
    if omit(fresh)!=omit(saved):raise ValueError('Typed SQL route values changed')
    return copy.deepcopy(saved)

manifest,jobs=checked_jobs69()
searches={r['identity']:r for r in T.read(BASE/'retrieval_results.json')['records']}
parsers={(r['query_id'],r['repeat']):r for r in T.read(BASE/'parser_results.json')['records']}
facts3=R.R.fact_map(db3);facts4=R.R.fact_map(db4)
# Real regression: fresh Pydantic dict key order versus saved sort_keys JSON.
probe=next(j for j in jobs if j['mode']=='SEQ_REPAIRED'and len(j['result']['evidence'])>len(searches[j['identity']]['result']['evidence']))
row=searches[probe['identity']];parser=parsers[probe['query_id'],probe['repeat']]
fresh=T.K.route(R,parser['plan'],row['original_scope'],db4,facts4,matcher='projected',parser_failed=parser['status']!='ok')
first=F.assemble(probe['question'],row['result'],fresh,facts4)['results']['B']
fixed=F.assemble(probe['question'],row['result'],saved_route_for_replay69(fresh,row['routing']),facts4)['results']['B']
assert first!=probe['result']and fixed==probe['result']
prefix=len(row['result']['evidence'])
assert all(json.loads(a['text'])==json.loads(b['text'])for a,b in zip(first['evidence'][prefix:],fixed['evidence'][prefix:],strict=True))
negative_checks=[]
for kind in('changed_numeric_condition','changed_witness'):
    bad=copy.deepcopy(fresh)
    if kind=='changed_numeric_condition':bad['plan']['groups'][0][0]['max_spend_krw']=300001
    else:bad['matching_fact_ids']={}
    try:saved_route_for_replay69(bad,row['routing'])
    except ValueError:negative_checks.append(kind)
    else:raise AssertionError('Changed route was accepted')
assert len(negative_checks)==2
freeze69(BASE/'verification_render_order_diagnostic.json',dict(status='reproduced_and_repaired_verification_only_encoding_mismatch',
    identity=probe['identity'],typed_route_values_equal=True,appended_JSON_values_equal=True,
    fresh_model_key_order_render_exact=False,saved_route_render_exact=True,
    negative_mutations_rejected=negative_checks,original_jobs_and_payloads_modified=False,
    original_verification_cell_sha256=R.P.sha(''.join(next(c for c in T.read(NOTEBOOK)['cells']if c['id']=='exp69-exec-07')['source'])),
    jobs_raw_sha256=raw69(BASE/'jobs.json'),payload_contract_raw_sha256=raw69(BASE/'answer_payload_contract.json')))

start=time.perf_counter();cpu=time.process_time();verified_routes=0
for job in jobs:
    row=searches[job['fixed_search_identity']]
    assert job['retrieval_row_sha256']==R.P.sha(row)
    disk=BASE/'retrievals'/(row['identity']+'.json')
    assert T.read(disk)==row and T.read(disk.with_suffix('.seal.json'))['row_sha256']==R.P.sha(row)
    R.G.T.S.validate_scope(job['result'],row['original_scope']['card_keys']);R.G.T.S.validate_scope(job['result'],row['routing']['scope'])
    parser=parsers[job['query_id'],job['repeat']]
    if job['mode']=='U':expected=row['result']
    else:
        repaired=job['mode']=='SEQ_REPAIRED';directory,fmap=(db4,facts4)if repaired else(db3,facts3)
        fresh=T.K.route(R,parser['plan'],row['original_scope'],directory,fmap,matcher='projected'if repaired else'baseline',parser_failed=parser['status']!='ok')
        verified=saved_route_for_replay69(fresh,row['routing']);verified_routes+=1
        expected=F.assemble(job['question'],row['result'],verified,facts4)['results'][config['profiles'][job['profile']]['input_arm']]if repaired else row['result']
    assert job['result']==expected,job['identity']
inventory=T.read(BASE/'additional_fact_inventory.json')
reconstructed_sha=validate_additional_inventory69(jobs,list(searches.values()),facts4,inventory)
freeze69(BASE/'planned_input_verification.json',dict(status='all432_planned_payloads_and_routes_verified_not_source_truth_acceptance',
    logical_jobs=432,shared_parser_rows=72,reconstructed_additional_inventory_sha256=reconstructed_sha,source_truth_gate_pending=True,manual_gold_in_assembler=False,
    jobs_raw_sha256=raw69(BASE/'jobs.json'),retrieval_results_raw_sha256=raw69(BASE/'retrieval_results.json'),
    payload_contract_raw_sha256=raw69(BASE/'answer_payload_contract.json'),additional_fact_inventory_raw_sha256=raw69(BASE/'additional_fact_inventory.json'),
    typed_SQL_replayed_both_DB_versions=True,original_scope_preserved=True,original_U_prefix_preserved=True,global_U_union=False,
    verification_version='saved_JSON_route_after_typed_value_equivalence69_v1',
    verification_code_sha256=R.P.sha(''.join(next(c for c in T.read(NOTEBOOK)['cells']if c['id']=='exp69-verify-saved-route-01')['source'])),
    verification_diagnostic_raw_sha256=raw69(BASE/'verification_render_order_diagnostic.json')))
T.write(BASE/'input_verification_resources.json',resources69(start,cpu,checked_jobs=432,typed_SQL_replays=verified_routes))
print('동일 typed SQL 값과 저장된 직렬화 순서로432입력 정확 재생. 원래07의 encoding 재현오류만 보완, 유료phase/원입력 변경0.')


동일 typed SQL 값과 저장된 직렬화 순서로432입력 정확 재생. 원래07의 encoding 재현오류만 보완, 유료phase/원입력 변경0.


## 원문 gate 거절로 종료
원래 검색·입력·감사 결과를 보존한다. 답변432건은 실행하지 않았으며70번 사후 수정 개발 실험으로 분리한다. 보호된 유료 실행 셀과 원래07 실패는 수정하지 않는다.


In [70]:
# 원문 판정은 저장된 수동 기록이다. 이 셀은 값·출처·gate 결속과 집계만 검사한다.
from collections import Counter
I69=module69('source_integrity69_close','68_review_integrity.py')
source=T.read(BASE/'source_review/source_review.json');T.E.verify_file_map(source['files_sha256'],ROOT,fresh=True)
for proof in source['source_proof_catalog'].values():I69.V.validate_source_support(proof,ROOT)
assert source['coverage']['reviewed_unique_facts']==55
assert len(source['confirmed_source_conflicts'])==1
assert source['source_gate_accepted'] is False
check=T.read(BASE/'source_gate_runtime_check.json')
assert check['answer_rows']==0 and check['next_answer_cell_executed'] is False
assert not list((BASE/'answers').glob('*.json'))
assert T.phase_code_sha(NOTEBOOK,'exp69-exec-')==check['original_paid_phase_sha256']
rows=T.read(BASE/'retrieval_results.json')['records'];packages=T.read(BASE/'packages.json')['records']
conditions=[]
for profile in PROFILES:
 for mode in MODES:
  selected=[r for r in rows if r['profile']==profile and r['mode']==mode]
  assert len(selected)==72
  conditions.append(dict(profile=profile,mode=mode,logical_searches=72,technical_ok=sum(r['retrieval_status']=='ok'for r in selected),answer_generation='unexecuted'))
ledger=T.budget_ledger();assert ledger['unknown_cost_records']==0
assert ledger['budget_debit_usd']=='2.168031040'
summary=dict(status='closed_rejected_source_gate_before_answer_comparison',conditions=conditions,
 package_reasons=dict(Counter(r['reason']for r in packages)),source_coverage=source['coverage'],
 confirmed_source_conflict_facts=len(source['confirmed_source_conflicts']),relationship_ambiguous_facts=len(source['relationship_ambiguous']),
 actual_answer_API_calls=0,planned_answers_unexecuted=432,new_stage_usage_rate_cost_usd='0.009729165',
 cumulative_budget_debit_usd=ledger['budget_debit_usd'],remaining_under8_usd='5.831968960',
 invoice_cost_usd=None,independent_holdout=False,production_promotion=False,
 source_review_raw_sha256=raw69(BASE/'source_review/source_review.json'),
 report_path='docs/report/69_repaired_sequential_confirmation_report.html',
 original_protected_phase_sha256=check['original_paid_phase_sha256'],
 followup='70 posthoc source-repair development experiment; original69 preserved',
 full_notebook_RunAll=False,original07_ordering_failure_preserved=True)
T.write(BASE/'closure_summary.json',summary)
print('69 종료: 검색432개 준비/추가사실55개 원문검토/확정충돌1개/gate거절/답변0. 비용 $0.009729165. 답변 비교 미실행.')


69 종료: 검색432개 준비/추가사실55개 원문검토/확정충돌1개/gate거절/답변0. 비용 $0.009729165. 답변 비교 미실행.
